# rai-microrts-arena Kaggle T4x2 ?????

?? notebook ?? Kaggle ? T4 ????????????? microRTS ?????????????????????

1. ?? GPU/Java/Python ???
2. ????? microRTS ???
3. ?? Kaggle ???????
4. ? profile `squeeze_unet` ? `hybrid_entity_grid`?
5. ??????? baseline?
6. ?? CPU ?? Java ???????? T4 ????? seed/?????

> ????? PPO ???? DDP ???T4x2 ???????????????????? GPU0 ? SquNet baseline?GPU1 ? HybridEntityGrid ???

## ????????

**??/????**?`eval/score`?`WinLoss`?? Mayari/Coac/WorkerRush/LightRush ??????????? shaped reward????????????

**PPO ???**?`approx_kl`?`clip_fraction`?`entropy`?`explained_variance`?`policy_loss`?`value_loss`?`grad_norm`???? KL ???? 0 ?????????????????entropy ???????????????

**?? mask/????**?`action_mask_stats/valid_locs`?`action_mask_stats/no_valid`?episode length?timeout/truncation?`no_valid` ???? 0?valid locs ???????? mask ?????????

**?????**?`steps_per_second`?GPU util?GPU memory?`scripts/profile_microrts_policy.py` ? `p95_ms`?Kaggle ? Java env/CPU ??? GPU ????????

**????**?`n_envs`?`n_steps`?`batch_size`?`n_epochs`?`learning_rate`?`clip_range`?`ent_coef`?`vf_coef`?`gamma`?`gae_lambda`?????????? `learning_rate` ? `n_epochs`?????????? entropy ?? `ent_coef`?

In [ ]:
# 0. Runtime sanity check
import os, sys, subprocess, textwrap, json
from pathlib import Path

print(sys.version)
print('Kaggle working dir:', Path('/kaggle/working').exists())
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
# 1. User config
REPO_URL = 'https://github.com/SShion0721/rai-microrts-arena.git'
BRANCH = 'main'
WORKDIR = Path('/kaggle/working/rai-microrts-arena')

# Start small. Increase to 10e6/20e6 only after the smoke run is healthy.
KAGGLE_TIMESTEPS = '2e6'
WANDB_PROJECT = 'rai-microrts-kaggle'
USE_WANDB = False

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['WORKDIR'] = str(WORKDIR)
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
print('WORKDIR =', WORKDIR)
print('WANDB_MODE =', os.environ['WANDB_MODE'])


In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq default-jdk xvfb ffmpeg git unzip > /dev/null

if [ ! -d "$WORKDIR/.git" ]; then
  git clone --branch "$BRANCH" --depth 1 "$REPO_URL" "$WORKDIR"
else
  cd "$WORKDIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull --ff-only || true
fi

cd "$WORKDIR"
python -m pip install -U -q pip
# Avoid gymnasium[box2d] here; microRTS does not need Box2D and it can slow/fail installs.
python -m pip install -q -e . --no-deps
python -m pip install -q \
  'numpy<2' \
  'gymnasium==0.29.1' \
  'stable-baselines3[extra]==2.1.0' \
  wandb tensorboard 'ray[air]>=2.8.1' accelerate einops GPUtil \
  pyvirtualdisplay moviepy JPype1 peewee 'PettingZoo==1.24.3' PyYAML tqdm psutil


In [ ]:
# 2. Import check
os.chdir(WORKDIR)
import gymnasium, stable_baselines3, yaml, jpype, ray
import rl_algo_impls
print('cwd:', Path.cwd())
print('gymnasium:', gymnasium.__version__)
print('stable_baselines3:', stable_baselines3.__version__)
print('java:')
!java -version


In [ ]:
# 3. Append Kaggle-friendly PPO configs. This only changes the notebook runtime copy.
from pathlib import Path

hp_path = WORKDIR / 'rl_algo_impls' / 'hyperparams' / 'ppo-Microrts.yml'
text = hp_path.read_text(encoding='utf-8')

KAGGLE_CONFIG = f'''

# ---- Kaggle T4x2 smoke/ablation configs appended by kaggle_microrts_t4x2.ipynb ----
Microrts-kaggle-squnet-map16-bots: &microrts-kaggle-squnet-map16-bots
  <<: *microrts-squnet-map16
  n_timesteps: !!float {KAGGLE_TIMESTEPS}
  evaluate_after_training: true
  env_hyperparams:
    <<: *microrts-squnet-map16-env-defaults
    n_envs: 12
    self_play_kwargs: null
    map_paths:
      - maps/16x16/basesWorkers16x16A.xml
      - maps/16x16/TwoBasesBarracks16x16.xml
      - maps/16x16/melee16x16Mixed12.xml
    make_kwargs:
      <<: *microrts-squnet-map16-env-make-kwargs-defaults
      num_selfplay_envs: 0
      num_bot_envs: 12
      max_steps: 3000
    bots:
      coacAI: 6
      mayari: 6
  rollout_hyperparams:
    <<: *microrts-ai-rollout-defaults
    n_steps: 512
  algo_hyperparams:
    <<: *microrts-squnet-map16-algo-defaults
    batch_size: 3072
    n_epochs: 4
    learning_rate: !!float 1e-4
    clip_range: 0.1
    ent_coef: 0.01
  eval_hyperparams:
    <<: *microrts-squnet-map16-eval-defaults
    step_freq: !!float 2.5e5
    n_episodes: 12
    env_overrides:
      <<: *microrts-squnet-map16-eval-env-overrides
      n_envs: 12
      self_play_kwargs: {{}}
      bots:
        coacAI: 3
        mayari: 3
        workerRushAI: 3
        lightRushAI: 3

Microrts-kaggle-hybrid-map16-bots:
  <<: *microrts-kaggle-squnet-map16-bots
  policy_hyperparams:
    <<: *microrts-squnet-map16-policy-defaults
    actor_head_style: hybrid_entity_grid
    normalization: layer
    encoder_embed_dim: 128
    encoder_attention_heads: 4
    encoder_feed_forward_dim: 256
    encoder_layers: 2
    actor_head_kernel_size: 3
'''.strip() + '
'

if 'Microrts-kaggle-squnet-map16-bots' not in text:
    hp_path.write_text(text.rstrip() + '

' + KAGGLE_CONFIG, encoding='utf-8')
    print('Appended Kaggle configs to', hp_path)
else:
    print('Kaggle configs already present')

print('Configured timesteps:', KAGGLE_TIMESTEPS)


In [ ]:
# 4. Fast architecture profiling before spending hours training
!python scripts/profile_microrts_policy.py \
  --styles squeeze_unet,hybrid_entity_grid \
  --map-size 16 --batch-size 4 --entities 24 \
  --warmup 5 --iters 20 \
  --action-mode sample


In [ ]:
# 5. Single-GPU baseline training. Start here.
# Watch TensorBoard/W&B while this runs. If it crashes, reduce n_envs in the appended config from 12 to 6.
!CUDA_VISIBLE_DEVICES=0 python train.py \
  --algo ppo \
  --env Microrts-kaggle-squnet-map16-bots \
  --seed 1 \
  --device-indexes 0 \
  --wandb-project-name $WANDB_PROJECT \
  --wandb-tags kaggle t4x2 squnet map16 bots


## ???? T4 ????

???????? CPU ?????????????????????????? PPO ??? GPU ?????? T4 ???????????

?????

- GPU0: `Microrts-kaggle-squnet-map16-bots`, seed 1
- GPU1: `Microrts-kaggle-hybrid-map16-bots`, seed 2

?? Kaggle CPU ??????????? `n_envs` ? `num_bot_envs` ? 12 ?? 6??? bots ?? `coacAI: 3`, `mayari: 3`?

In [ ]:
# 6. Optional dual-GPU parallel run: one experiment per T4.
# Set RUN_DUAL = True when you are ready.
RUN_DUAL = False

if RUN_DUAL:
    import subprocess, os, time
    log_dir = WORKDIR / 'kaggle_logs'
    log_dir.mkdir(exist_ok=True)
    jobs = [
        ('0', 'Microrts-kaggle-squnet-map16-bots', '1', 'squnet'),
        ('1', 'Microrts-kaggle-hybrid-map16-bots', '2', 'hybrid'),
    ]
    procs = []
    for gpu, env_name, seed, label in jobs:
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = gpu
        env['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
        log_path = log_dir / f'{label}-gpu{gpu}-seed{seed}.log'
        cmd = [
            sys.executable, 'train.py',
            '--algo', 'ppo',
            '--env', env_name,
            '--seed', seed,
            '--device-indexes', '0',
            '--wandb-project-name', WANDB_PROJECT,
            '--wandb-tags', 'kaggle', 't4x2', label, 'map16', 'bots',
        ]
        print('Starting', label, 'on visible GPU', gpu, 'log:', log_path)
        f = open(log_path, 'w')
        procs.append((label, subprocess.Popen(cmd, cwd=WORKDIR, env=env, stdout=f, stderr=subprocess.STDOUT), f))
    for label, proc, f in procs:
        code = proc.wait()
        f.close()
        print(label, 'exit code:', code)
else:
    print('RUN_DUAL is False; skipped.')


In [ ]:
# 7. TensorBoard
# In Kaggle, this may render inline. If it does not, use the runs/ folder as an output artifact.
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/rai-microrts-arena/runs


In [ ]:
# 8. Inspect and package outputs
os.chdir(WORKDIR)
!find saved_models -maxdepth 3 -type f | head -50 || true
!tar -czf /kaggle/working/rai_microrts_outputs.tgz saved_models runs videos kaggle_logs 2>/dev/null || true
print('Packed outputs to /kaggle/working/rai_microrts_outputs.tgz')


## ????????

- `approx_kl` ??????? `learning_rate` ?? `5e-5`??? `n_epochs` ? 4 ?? 2?
- `entropy` ???? 0???????????????? `ent_coef` ??? bot warmup?
- `explained_variance` ?????critic ?????? reward scale?`vf_coef`?`normalize_value_targets`?
- `steps_per_second` ??? GPU ??????CPU/Java env ?????? `n_envs` ?????????
- Hybrid ? SquNet ??????????????????/?????? Mamba?
- SquNet ? Hybrid ??????????????? ACBC warm-start?league/PFSP?GraphDINO ? Mamba entity branch?